# 11 — Integration diagnostics and writeback

Review sample/condition/QC patterns before interpreting integration. Annotations are attached to a zero-gene `cells_scvi` table without duplicating raw counts. Source inputs are never modified.
Writeback must occur BEFORE final immutable publication of per-sample objects. No forced batch-mixing or automatic cell-type annotation is performed.

In [ ]:
from pathlib import Path
import os, sys, json
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "vhd" / "control").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter in the extracted pipeline folder (or its notebooks folder).")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from vhd.control.manifest import load_project, check_policy, save_plan, sample_layout
MANIFEST = Path(os.environ.get("VHD_MANIFEST", ROOT / "config" / "samples.csv"))
SETTINGS = Path(os.environ.get("VHD_SETTINGS", ROOT / "config" / "settings.json"))
if not MANIFEST.exists() or not SETTINGS.exists():
    raise FileNotFoundError("Copy a supplied samples.*.csv to config/samples.csv and settings.g5_24xlarge.example.json to config/settings.json; edit paths and policy first.")
PROJECT = load_project(MANIFEST, SETTINGS)
check_policy(PROJECT)
# Default is a dry run. Set True here only after reviewing the printed plan.
EXECUTE = os.environ.get("VHD_EXECUTE", "0") == "1"
# Optional pilot selection, e.g. ["StudyLegacy__Sample01"]. None selects all applicable rows.
SAMPLE_KEYS = None


## Group and diagnostics

In [ ]:
groups = sorted({r["integration_group"] for r in PROJECT["rows"] if r["integration_group"]})
if not groups: raise ValueError("No integration group.")
GROUP = groups[0]
from vhd.compute.launch import launch_integration
launch_integration(PROJECT, GROUP, "diagnostics", execute=EXECUTE)

## Append the small annotation-only tables

In [ ]:
launch_integration(PROJECT, GROUP, "writeback", execute=EXECUTE)